# Validation: PR flash vs thermo

Compare chemthermo TP flash (Peng-Robinson) against `thermo` for a pure vapor case and a VLE
mixture case. Results are printed as compact tables and a final assertions cell enforces the same
tolerances as `tests/validation/test_flash_vs_thermo.py`.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path
import sys
from typing import Any

import numpy as np
import chemthermo as ct

try:
    import thermo
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires the 'thermo' package. Install with: pip install thermo"
    ) from exc


def _ensure_repo_root_on_path() -> None:
    cwd = Path.cwd()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "pyproject.toml").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return


_ensure_repo_root_on_path()

from notebooks._nb_utils import print_table

CEOSGas = thermo.CEOSGas
CEOSLiquid = thermo.CEOSLiquid
ChemicalConstantsPackage = thermo.ChemicalConstantsPackage
FlashPureVLS = thermo.FlashPureVLS
FlashVL = thermo.FlashVL
PRMIX = thermo.PRMIX

# Tolerances aligned with tests/validation/test_flash_vs_thermo.py
PURE_VF_ABS = 1e-6
VF_TOL = {"rel": 5e-2, "abs": 5e-3}
COMP_TOL = {"rel": 5e-2, "abs": 1e-2}


def _get_vf(state: Any) -> float | None:
    vf = getattr(state, "VF", None)
    if vf is None:
        return None
    vf_value = vf() if callable(vf) else vf
    if isinstance(vf_value, (int, float)):
        return float(vf_value)
    return None


def _pr_flasher(ids: list[str]) -> Any:
    constants, properties = ChemicalConstantsPackage.from_IDs(ids)
    kijs = [[0.0 for _ in ids] for _ in ids]
    eos_kwargs = {
        "Pcs": constants.Pcs,
        "Tcs": constants.Tcs,
        "omegas": constants.omegas,
        "kijs": kijs,
    }
    gas = CEOSGas(PRMIX, eos_kwargs=eos_kwargs, HeatCapacityGases=properties.HeatCapacityGases)
    liquid = CEOSLiquid(
        PRMIX, eos_kwargs=eos_kwargs, HeatCapacityGases=properties.HeatCapacityGases
    )
    return FlashVL(constants, properties, liquid=liquid, gas=gas)


def _phase_label_from_vf(vf: float | None) -> str:
    if vf is None:
        return "unknown"
    if math.isclose(vf, 0.0, abs_tol=1e-12):
        return "liquid"
    if math.isclose(vf, 1.0, abs_tol=1e-12):
        return "vapor"
    return "liquid+vapor"


In [ ]:
cases = [
    {
        "name": "pure_methane_vapor",
        "components": ["Methane"],
        "zs": [1.0],
        "temperature_K": 400.0,
        "pressure_Pa": 1.0e5,
        "expect_two_phase": False,
    },
    {
        "name": "methane_ethane_propane_vle",
        "components": ["Methane", "Ethane", "Propane"],
        "zs": [0.50, 0.30, 0.20],
        "temperature_K": 240.0,
        "pressure_Pa": 3.0e6,
        "expect_two_phase": True,
    },
]


In [ ]:
results: dict[str, dict[str, Any]] = {}

for case in cases:
    name = case["name"]
    components = case["components"]
    zs = case["zs"]
    temperature_K = case["temperature_K"]
    pressure_Pa = case["pressure_Pa"]

    mixture = ct.Mixture.from_database(components, zs, normalize=True)
    result = ct.flash_tp(
        mixture,
        temperature_K=temperature_K,
        pressure_Pa=pressure_Pa,
        eos=ct.PengRobinsonEOS(),
    )

    if len(components) == 1:
        constants, properties = ChemicalConstantsPackage.from_IDs([components[0].lower()])
        eos_kwargs = {
            "Pcs": constants.Pcs,
            "Tcs": constants.Tcs,
            "omegas": constants.omegas,
        }
        gas = CEOSGas(PRMIX, eos_kwargs=eos_kwargs, HeatCapacityGases=properties.HeatCapacityGases)
        liquid = CEOSLiquid(
            PRMIX, eos_kwargs=eos_kwargs, HeatCapacityGases=properties.HeatCapacityGases
        )
        flasher = FlashPureVLS(constants, properties, gas=gas, liquids=[liquid], solids=[])
        ref = flasher.flash(T=temperature_K, P=pressure_Pa)
    else:
        flasher = _pr_flasher([comp.lower() for comp in components])
        ref = flasher.flash(T=temperature_K, P=pressure_Pa, zs=zs)

    chem_phases = list(result.phase_names())
    chem_vf = result.vapor_fraction
    chem_x = result.phases["liquid"].composition.fractions if "liquid" in result.phases else None
    chem_y = result.phases["vapor"].composition.fractions if "vapor" in result.phases else None
    chem_k = (np.asarray(chem_y) / np.asarray(chem_x)) if chem_x is not None and chem_y is not None else None

    ref_vf = _get_vf(ref)
    ref_x = getattr(getattr(ref, "liquid0", None), "zs", None)
    ref_y = getattr(getattr(ref, "gas", None), "zs", None)
    ref_k = (np.asarray(ref_y) / np.asarray(ref_x)) if ref_x is not None and ref_y is not None else None

    results[name] = {
        "components": components,
        "zs": zs,
        "temperature_K": temperature_K,
        "pressure_Pa": pressure_Pa,
        "chem_phases": chem_phases,
        "chem_vf": chem_vf,
        "chem_x": chem_x,
        "chem_y": chem_y,
        "chem_k": chem_k,
        "ref_vf": ref_vf,
        "ref_x": ref_x,
        "ref_y": ref_y,
        "ref_k": ref_k,
    }

    print_table(
        [
            {
                "case": name,
                "components": components,
                "T [K]": temperature_K,
                "P [Pa]": pressure_Pa,
                "z": zs,
            }
        ],
        title="Input summary",
    )

    print_table(
        [
            {
                "source": "chemthermo",
                "phases": "+".join(chem_phases) if chem_phases else "-",
                "vapor_fraction": chem_vf,
                "x (liquid)": chem_x,
                "y (vapor)": chem_y,
                "K (y/x)": chem_k,
            },
            {
                "source": "thermo",
                "phases": _phase_label_from_vf(ref_vf),
                "vapor_fraction": ref_vf,
                "x (liquid)": ref_x,
                "y (vapor)": ref_y,
                "K (y/x)": ref_k,
            },
        ],
        title="Results",
    )

    if chem_vf is not None and ref_vf is not None:
        print_table(
            [
                {
                    "metric": "vapor_fraction",
                    "chemthermo": chem_vf,
                    "thermo": ref_vf,
                    "abs_diff": abs(chem_vf - ref_vf),
                    "rel_diff": abs(chem_vf - ref_vf) / max(abs(ref_vf), 1e-12),
                }
            ],
            title="Differences",
        )

    if chem_x is not None and ref_x is not None:
        rows = []
        for comp, cx, rx in zip(components, chem_x, ref_x):
            diff = float(cx) - float(rx)
            rel = diff / (abs(float(rx)) if abs(float(rx)) > 1e-12 else 1.0)
            rows.append({"component": comp, "chem_x": cx, "thermo_x": rx, "abs_diff": diff, "rel_diff": rel})
        print_table(rows, title="Liquid composition differences")

    if chem_y is not None and ref_y is not None:
        rows = []
        for comp, cy, ry in zip(components, chem_y, ref_y):
            diff = float(cy) - float(ry)
            rel = diff / (abs(float(ry)) if abs(float(ry)) > 1e-12 else 1.0)
            rows.append({"component": comp, "chem_y": cy, "thermo_y": ry, "abs_diff": diff, "rel_diff": rel})
        print_table(rows, title="Vapor composition differences")


In [ ]:
# Assertions (mirrors tests/validation/test_flash_vs_thermo.py)
pure = results["pure_methane_vapor"]
assert pure["chem_phases"] == ["vapor"], f"Expected vapor-only, got {pure['chem_phases']}"
assert pure["ref_vf"] is not None and abs(pure["ref_vf"] - 1.0) <= PURE_VF_ABS

mix = results["methane_ethane_propane_vle"]
chem_two_phase = "liquid" in mix["chem_phases"] and "vapor" in mix["chem_phases"]
ref_two_phase = mix["ref_vf"] is not None and 0.0 < mix["ref_vf"] < 1.0

if not (chem_two_phase and ref_two_phase):
    print("Skipping VLE assertions: single-phase result in at least one solver.")
else:
    vf_diff = abs(mix["chem_vf"] - mix["ref_vf"])
    vf_rel = vf_diff / max(abs(mix["ref_vf"]), 1e-12)
    assert vf_diff <= VF_TOL["abs"] or vf_rel <= VF_TOL["rel"], (
        f"Vapor fraction diff too large: abs={vf_diff:.3g}, rel={vf_rel:.3g}"
    )
    assert np.allclose(
        mix["chem_x"],
        mix["ref_x"],
        rtol=COMP_TOL["rel"],
        atol=COMP_TOL["abs"],
    )
    assert np.allclose(
        mix["chem_y"],
        mix["ref_y"],
        rtol=COMP_TOL["rel"],
        atol=COMP_TOL["abs"],
    )
